In [2]:
import subprocess
print(subprocess.run("ls -la /mnt/g/rice-leaf-disease-xai/data/", shell=True, capture_output=True, text=True).stdout)
print(subprocess.run("ls -la /mnt/g/rice-leaf-disease-xai/data/external/ 2>&1", shell=True, capture_output=True, text=True).stdout)

total 0
drwxrwxrwx 1 jaimul jaimul 4096 Sep 29 23:47 .
drwxrwxrwx 1 jaimul jaimul 4096 Sep 26 23:49 ..
drwxrwxrwx 1 jaimul jaimul 4096 Sep 25 21:08 Rice Dataset
drwxrwxrwx 1 jaimul jaimul 4096 Sep 30 00:07 external

total 0
drwxrwxrwx 1 jaimul jaimul 4096 Sep 30 00:07 .
drwxrwxrwx 1 jaimul jaimul 4096 Sep 29 23:47 ..
drwxrwxrwx 1 jaimul jaimul 4096 Sep 29 23:50 RiceLeafBD
drwxrwxrwx 1 jaimul jaimul 4096 Sep 30 00:05 Sethy



In [3]:
import sys, numpy as np, torch, torch.nn as nn
from pathlib import Path
from collections import Counter
from torchvision import models
from torch.utils.data import DataLoader
from sklearn.metrics import confusion_matrix
sys.path.append("../src")
from dataset import RiceLeafDataset, CLASSES

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
EXT = {
    "RiceLeafBD": Path("/mnt/g/rice-leaf-disease-xai/data/external/RiceLeafBD"),
    "Sethy":      Path("/mnt/g/rice-leaf-disease-xai/data/external/Sethy"),
}
IMG_EXT = {".jpg", ".jpeg", ".png", ".bmp"}

def scan(root):
    c = Counter()
    for f in root.rglob("*"):
        if f.suffix.lower() in IMG_EXT:
            c[f.parent.name] += 1
    return c

print("Our CLASSES:", CLASSES)
for k, p in EXT.items():
    print(f"\n{k}: exists={p.exists()}")
    for name, cnt in sorted(scan(p).items()):
        print(f"   {name!r}: {cnt}")

Our CLASSES: ['Healthy', 'Insect', 'Leaf Scald', 'Rice Blast', 'Rice Leaffolder', 'Rice Stripes', 'Rice Tungro']

RiceLeafBD: exists=True
   'Bacterialblight': 1584
   'Blast': 1440
   'Brownspot': 1600
   'Tungro': 1308

Sethy: exists=True
   'Bacterial Leaf Blight': 422
   'Brown Spot': 356
   'Healthy Leaf': 252
   'Tungro Virus': 530


In [ ]:
MAP = {
    "RiceLeafBD": {"Blast": "Rice Blast", "Tungro": "Rice Tungro"},
    "Sethy":      {"Healthy Leaf": "Healthy", "Tungro Virus": "Rice Tungro"},
}
for d in MAP.values():
    for v in d.values(): assert v in CLASSES, f"{v} not in CLASSES"

CKPT = next(p for p in ["convnext_tiny_best.pth", "../notebooks/convnext_tiny_best.pth",
                        "../convnext_tiny_best.pth"] if Path(p).exists())
model = models.convnext_tiny(weights=None)
model.classifier[2] = nn.Linear(model.classifier[2].in_features, len(CLASSES))
model.load_state_dict(torch.load(CKPT, map_location=device))
model = model.to(device).eval()
print("Loaded:", CKPT)

def run(name):
    samples = [(str(f), CLASSES.index(MAP[name][f.parent.name]))
               for f in EXT[name].rglob("*")
               if f.suffix.lower() in IMG_EXT and f.parent.name in MAP[name]]
    loader = DataLoader(RiceLeafDataset(samples, split="test"), batch_size=32, num_workers=4)
    logits, gts = [], []
    with torch.no_grad():
        for x, y in loader:
            logits.append(model(x.to(device)).cpu()); gts += y.tolist()
    logits = torch.cat(logits); gts = np.array(gts)
    preds = logits.argmax(1).numpy()

    keep = np.array(sorted(set(gts.tolist())))
    restricted = keep[logits[:, keep].argmax(1).numpy()]
    print(f"\n===== {name} | n={len(gts)} =====")
    print(f"7-way accuracy       : {(preds == gts).mean():.4f}")
    print(f"restricted accuracy  : {(restricted == gts).mean():.4f}  (শুধু {[CLASSES[i] for i in keep]})")
    for i in keep:
        m = gts == i
        print(f"  recall {CLASSES[i]:<12}: {(preds[m] == i).mean():.4f} | restricted {(restricted[m] == i).mean():.4f} (n={m.sum()})")
    print("  prediction distribution:", {CLASSES[i]: int((preds == i).sum()) for i in range(len(CLASSES))})
    print("  confusion rows (true) x 7 pred cols:\n", confusion_matrix(gts, preds, labels=list(range(len(CLASSES))))[keep])

for name in EXT: run(name)

Loaded: convnext_tiny_best.pth
